# AGILAB API first run in Kaggle

This notebook is the smallest Kaggle path for the built-in MyCode example app (`mycode_project`).
It prepares an isolated Kaggle runtime venv under `/kaggle/working`, installs the published core/runtime packages there, and runs one local AGILAB execution through `AgiEnv` and `AGI.run(...)` without mutating the base Kaggle kernel packages.

What to expect:
- Cell 1 prepares the isolated Kaggle runtime.
- Cell 2 imports AGILAB and shows which built-in example app will run.
- Cell 3 installs the worker if needed and executes one local run.

Kaggle Internet must be enabled for the first install.
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ThalesGroup/agilab/blob/main/src/agilab/examples/notebook_quickstart/agi_core_kaggle_first_run.ipynb)


In [ ]:
# Published-package Kaggle path: prepare an isolated Kaggle runtime venv.
import json
import socket
import subprocess
import sys
import shutil
from pathlib import Path

VENV_ROOT = Path("/kaggle/working/.agilab_kaggle_pypi_env")
STATE_FILE = Path("/kaggle/working/.agilab_kaggle_pypi_state.json")
STATE_VERSION = 3


def require_kaggle_internet(*hosts):
    missing = []
    for host in hosts:
        try:
            socket.getaddrinfo(host, 443)
        except OSError:
            missing.append(host)
    if missing:
        hosts_text = ", ".join(missing)
        raise SystemExit(
            f"Kaggle Internet appears to be disabled for this notebook session; cannot reach {hosts_text}. Enable Internet in the Kaggle notebook settings, then rerun Cell 1."
        )


def ensure_uv():
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", "uv"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)


def venv_python(venv_root: Path) -> Path:
    return venv_root / "bin" / "python"


def purelib_for(venv_python_path: Path) -> str:
    return subprocess.run(
        [
            str(venv_python_path),
            "-c",
            "import sysconfig; print(sysconfig.get_paths()['purelib'])",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


require_kaggle_internet("pypi.org")

state = {}
if STATE_FILE.is_file():
    try:
        state = json.loads(STATE_FILE.read_text())
    except Exception:
        state = {}

venv_python_path = venv_python(VENV_ROOT)
needs_install = (
    state.get("version") != STATE_VERSION
    or state.get("mode") != "pypi"
    or not venv_python_path.is_file()
)

if needs_install:
    ensure_uv()
    shutil.rmtree(VENV_ROOT, ignore_errors=True)
    subprocess.run([
        "uv", "venv", "--system-site-packages", str(VENV_ROOT), "--python", sys.executable,
    ], check=True)
    venv_python_path = venv_python(VENV_ROOT)
    subprocess.run([
        "uv", "pip", "install", "--python", str(venv_python_path), "-q", "agi-core", "agi-apps",
    ], check=True)
    subprocess.run([
        "uv", "pip", "install", "--python", str(venv_python_path), "-q", "--no-deps", "agilab",
    ], check=True)
    STATE_FILE.write_text(json.dumps({
        "version": STATE_VERSION,
        "mode": "pypi",
        "venv_root": str(VENV_ROOT),
        "venv_python": str(venv_python_path),
        "purelib": purelib_for(venv_python_path),
    }))
    print("AGILAB PyPI runtime prepared in an isolated Kaggle venv.")
else:
    print("AGILAB PyPI runtime already prepared in an isolated Kaggle venv.")


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

STATE_FILE = Path("/content/.agilab_colab_pypi_state.json")
if not STATE_FILE.is_file():
    raise RuntimeError("Run Cell 1 first to prepare the isolated Colab PyPI runtime.")
try:
    _colab_state = json.loads(STATE_FILE.read_text())
except Exception as exc:
    raise RuntimeError("Colab PyPI runtime state is unreadable. Rerun Cell 1.") from exc

PURELIB = _colab_state.get("purelib")
if not PURELIB:
    raise RuntimeError("Colab PyPI runtime metadata is missing `purelib`. Rerun Cell 1.")
if PURELIB not in sys.path:
    sys.path.insert(0, PURELIB)

os.environ["PYTHONNOUSERSITE"] = "1"
os.environ.pop("PYTHONSTARTUP", None)

for bad_prefix in ("/content/agilab/src", "/content/agilab"):
    sys.path = [p for p in sys.path if p != bad_prefix and not p.startswith(bad_prefix + "/")]
for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

from agi_cluster.agi_distributor import AGI
from agilab.notebook_demo import (
    install_if_needed,
    notebook_app_env,
    notebook_local_request,
    notebook_log_root,
    resolve_notebook_apps_path,
)

APP = "mycode_project"
APPS_PATH = resolve_notebook_apps_path(APP)
LOG_ROOT = notebook_log_root(APP)

print("Apps path:", APPS_PATH)
print("App:", APP)
print("Log root:", LOG_ROOT)


In [ ]:
app_env = notebook_app_env(APP, apps_path=APPS_PATH, verbose=0)
request = notebook_local_request()
await install_if_needed(app_env, request=request)
result = await AGI.run(app_env, request=request)
result
